# XR Client Cursors

Examples of responding to generic user controller interaction.

## Setup runner & utilities

In [1]:
from nanover.app import OmniRunner
from nanover.openmm import OpenMMSimulation

simulation = OpenMMSimulation.from_xml_path("../openmm/openmm_files/17-ala.xml")
simulation.load()

imd_runner = OmniRunner.with_basic_server(simulation, port=0, name="EXAMPLE: xr client cursors")
imd_runner.load(0)

In [2]:
from nanover.jupyter import NanoverJupyterUtilities

utilities = NanoverJupyterUtilities.from_runner(imd_runner)

In [3]:
import numpy as np
from nanover.jupyter import Mode


OBJECT_RADIUS = .2
OBJECT_COLOR = [1.0, 0.0, 0.0, 1.0]
HOVER_COLOR = [1.0, 1.0, 1.0, 0.5]
CURSOR_RADIUS = .5

# mapping of cursor to currently grabbed object if any
CURSOR_GRABBED_OBJECT_ID = {}
# mapping of object id to object position
OBJECT_POSITIONS = {}


def spline_4p(t, p_1, p0, p1, p2):
    # https://stackoverflow.com/questions/1251438/catmull-rom-splines-in-python
    # wikipedia Catmull-Rom -> Cubic_Hermite_spline
    # 0 -> p0,  1 -> p1,  1/2 -> (- p_1 + 9 p0 + 9 p1 - p2) / 16
    # assert 0 <= t <= 1
    return (
          t*((2-t)*t - 1)     * p_1
        + (t*t*(3*t - 5) + 2) * p0
        + t*((4 - 3*t)*t + 1) * p1
        + (t-1)*t*t           * p2 ) / 2


def redraw_spline():
    try:
        P = [np.array(OBJECT_POSITIONS[str(i)]) for i in range(len(OBJECT_POSITIONS))]
        P = [P[0], *P, P[-1]]  # padding
        points = []

        for j in range(1, len(P)-2): # skip the ends
            for t in range(10):  # t: 0 .1 .2 .. .9
                p = spline_4p(t/10, P[j-1], P[j], P[j+1], P[j+2])
                points.append(p)

        utilities.objects.update_line("spline", positions=points, color=OBJECT_COLOR, size=0.05)
    except Exception as e:
        utilities.notify_all(str(e))


def redraw_object(object_id):
    utilities.objects.update_shape(object_id, position=OBJECT_POSITIONS[object_id], color=OBJECT_COLOR, size=OBJECT_RADIUS)


def intersect_objects(position):
    for object_id, object_pos in OBJECT_POSITIONS.items():
        if np.linalg.norm(np.subtract(position, object_pos)) < OBJECT_RADIUS + CURSOR_RADIUS:
            return object_id
    return None


# put some objects in the scene
for i in range(4):
    OBJECT_POSITIONS[str(i)] = [i, i, i]
    redraw_object(str(i))


class MoveObjectMode(Mode):
    def on_button_pressed(self, *, key: str, cursor: dict, button: str):
        next_pos = utilities.scene_transform.point_parent_to_local(cursor["position"])
        hovered = intersect_objects(next_pos)
        available = hovered not in CURSOR_GRABBED_OBJECT_ID.values()

        # grab hovered object if not already grabbed
        if button == "primary" and available:
            CURSOR_GRABBED_OBJECT_ID[key] = hovered

    def on_button_released(self, *, key: str, cursor: dict, button: str):
        # release grabbed
        if button == "primary":
            CURSOR_GRABBED_OBJECT_ID.pop(key, None)

    def on_cursor_updated(self, *, key: str, cursor: dict):
        next_pos = utilities.scene_transform.point_parent_to_local(cursor["position"])

        # if this cursor has grabbed something, update its position
        if key in CURSOR_GRABBED_OBJECT_ID:
            object_id = CURSOR_GRABBED_OBJECT_ID[key]
            OBJECT_POSITIONS[object_id] = next_pos
            redraw_object(object_id)
            redraw_spline()

        # show/remove hover graphic is this cursor hovers something
        hovered = intersect_objects(next_pos)
        if hovered is None:
            utilities.objects.update_shape(f"hovered.{key}")
        else:
            utilities.objects.update_shape(f"hovered.{key}", position=OBJECT_POSITIONS[hovered], color=HOVER_COLOR, size=OBJECT_RADIUS * 1.1)


utilities.use_interaction_modes()
utilities.add_interaction_mode(MoveObjectMode, "move handle")